# GEC Pipeline - Training and Inference

This notebook walks through the complete GEC (Grammatical Error Correction) pipeline:
1. Data preparation and feature extraction
2. Training the edit tagger model
3. Running inference on new text

## Setup

In [15]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

print("Setup complete!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Setup complete!


## Part 1: Data Preparation

In [16]:
from src.services.gec.config import (
    TRAIN_SENT_PATH,
    TRAIN_COR_PATH,
    NOPNX_TRAIN_OUTPUT,
    PNX_TRAIN_OUTPUT,
    LABEL2ID_PATH,
    ID2LABEL_PATH,
    CHECKPOINT_PATH,
)

print("Checking data files...")
print(f"Training sentences: {TRAIN_SENT_PATH.exists()}")
print(f"Training corrections: {TRAIN_COR_PATH.exists()}")
print(f"Checkpoint (processed data): {CHECKPOINT_PATH.exists()}")

Checking data files...
Training sentences: True
Training corrections: True
Checkpoint (processed data): False


In [17]:
import json

from src.services.gec.features.build_train import build_train

if CHECKPOINT_PATH.exists():
    print("Loading existing processed data...")
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        num_lines = sum(1 for _ in f)
    print(f"✓ Found {num_lines} training examples")
    
    with open(LABEL2ID_PATH, 'r', encoding='utf-8') as f:
        label2id = json.load(f)
    print(f"✓ Label vocabulary: {len(label2id)} labels")
else:
    print("⚠️  No processed data found. Run build_train() first.")
    build_train()

⚠️  No processed data found. Run build_train() first.
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
26

In [18]:
print("Sample training example:")
with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    example = json.loads(first_line.strip())
    print(f"Subwords (first 15): {example['subwords'][:15]}")
    print(f"Labels (first 15): {example['labels'][:15]}")
    print(f"Total subwords: {len(example['subwords'])}")
    print(f"Total labels: {len(example['labels'])}")
    print(f"Length match: {len(example['subwords']) == len(example['labels'])}")

Sample training example:
Subwords (first 15): ['الى', 'التعليق', 'رقم', '2', ' ', 'اك', '##يد', 'ان', 'لحكام', 'العرب', 'والمسلمين', 'مسؤولية', 'يتمثل', 'اد', '##ناها']
Labels (first 15): ['R_[إ]K2', 'K7', 'K3', 'K', 'R_[:]', 'R_[أ]K', 'K2', 'R_[أ]K', 'I_[ل]K5', 'K5', 'K9', 'K7', 'K5', 'R_[أ]K', 'K4']
Total subwords: 56
Total labels: 56
Length match: True


## Part 2: Model Training

In [19]:
from transformers import AutoTokenizer
from src.services.gec.training.datasets import GECTrainingDataset
from src.services.gec.training.model import GECTaggerModel
from src.services.gec.training.trainer import build_trainer

MODEL_CHECKPOINT = "aubmindlab/bert-base-arabertv02"
OUTPUT_DIR = Path("./gec_models/edit_tagger_v1")
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
MAX_LENGTH = 256

print(f"Model: {MODEL_CHECKPOINT}")
print(f"Output: {OUTPUT_DIR}")
print(f"Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, Max length: {MAX_LENGTH}")

Model: aubmindlab/bert-base-arabertv02
Output: gec_models/edit_tagger_v1
Epochs: 3, Batch: 8, Max length: 256


In [20]:
with open(LABEL2ID_PATH, 'r', encoding='utf-8') as f:
    label2id = json.load(f)

with open(ID2LABEL_PATH, 'r', encoding='utf-8') as f:
    id2label = json.load(f)

print(f"Labels: {len(label2id)}")
print(f"\nMost common labels (first 30):")
for i, (label, idx) in enumerate(list(label2id.items())[:30]):
    print(f"  {idx}: {label}")

Labels: 4485

Most common labels (first 30):
  0: [PAD]
  1: [UNK_EDIT]
  2: D
  3: D2
  4: D2K
  5: D2K2
  6: D2K2D2
  7: D2K2D2K
  8: D2K2R_[حفظ]
  9: D2K3
  10: D2KD
  11: D2KD2
  12: D2KD2R_[ع]
  13: D2KD2R_[ن]
  14: D2KD3
  15: D2KD3R_[ا]
  16: D2KD3R_[لى]
  17: D2KD3R_[ن]
  18: D2KDKR_[أ]KDK
  19: D2KDR_[أرى]
  20: D2KDR_[ا]
  21: D2KDR_[بهة]
  22: D2KDR_[فس]
  23: D2KDR_[ل]
  24: D2KDR_[ل]KD4
  25: D2KDR_[ن]
  26: D2KDR_[ن]K2R_[ت]
  27: D2KR_[ا]
  28: D2KR_[د]
  29: D2KR_[دء]


In [21]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"✓ Vocab size: {tokenizer.vocab_size}")
print(f"✓ PAD token: {tokenizer.pad_token}")
print(f"✓ PAD token ID: {tokenizer.pad_token_id}")

Loading tokenizer...
✓ Vocab size: 64000
✓ PAD token: [PAD]
✓ PAD token ID: 0


In [22]:
print("Loading training dataset...")
train_dataset = GECTrainingDataset(
    jsonl_path=CHECKPOINT_PATH,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=MAX_LENGTH,
)
print(f"✓ Training examples: {len(train_dataset)}")

sample = train_dataset[0]
print(f"\nSample item structure:")
print(f"  input_ids length: {len(sample['input_ids'])}")
print(f"  attention_mask length: {len(sample['attention_mask'])}")
print(f"  labels length: {len(sample['labels'])}")
print(f"  Lengths match: {len(sample['input_ids']) == len(sample['labels'])}")

Loading training dataset...
✓ Training examples: 501

Sample item structure:
  input_ids length: 80
  attention_mask length: 80
  labels length: 80
  Lengths match: True


In [23]:
print("Initializing model...")
model = GECTaggerModel(
    checkpoint=MODEL_CHECKPOINT,
    label2id=label2id,
)
print(f"✓ Model initialized with {len(label2id)} output labels")

Initializing model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 291.83it/s]
[transformers] BertForTokenClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arc

✓ Model initialized with 4485 output labels


In [24]:
print("Building trainer...")
trainer = build_trainer(
    model_wrapper=model,
    train_dataset=train_dataset,
    eval_dataset=None,
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    label2id_path=LABEL2ID_PATH,
    id2label_path=ID2LABEL_PATH,
)
print("✓ Trainer ready!")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Building trainer...


✓ Trainer ready!


In [25]:
print("\n🚀 Starting training...")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("✓ Training complete!")


🚀 Starting training...


ValueError: expected sequence of length 64 at dim 1 (got 96)

In [ ]:
best_model_path = OUTPUT_DIR / "best"
print(f"Saving model to {best_model_path}...")

trainer.save_model(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

print(f"✓ Model saved")
print(f"\nFiles:")
if best_model_path.exists():
    for f in best_model_path.iterdir():
        print(f"  - {f.name}")

## Part 3: Inference

In [ ]:
import torch
from transformers import AutoModelForTokenClassification
from src.services.gec.modules.edit_tagger.inference import GECInferencePipeline
from src.services.gec.utils.string_utils import Tokenizer
from src.services.gec.schemas import GECInput, Token

if not best_model_path.exists():
    print("⚠️  Model not found. Please train first.")
else:
    print(f"Loading model from {best_model_path}...")
    inference_tokenizer = AutoTokenizer.from_pretrained(best_model_path)
    inference_model = AutoModelForTokenClassification.from_pretrained(best_model_path)
    print(f"✓ Model loaded ({inference_model.config.num_labels} labels)")

In [ ]:
class LabelVocab:
    def __init__(self, id2label):
        self.id2label = id2label

class DummyRewriter:
    pass

if best_model_path.exists():
    custom_tokenizer = Tokenizer()
    label_vocab = LabelVocab(id2label)
    rewriter = DummyRewriter()

    pipeline = GECInferencePipeline(
        model=inference_model,
        tokenizer=custom_tokenizer,
        label_vocab=label_vocab,
        rewriter=rewriter,
    )
    print("✓ Inference pipeline ready!")

In [ ]:
def create_gec_input(text: str) -> GECInput:
    words = text.split()
    tokens = [Token(form=word, start=i, end=i+len(word)) for i, word in enumerate(words)]
    return GECInput(
        text=text,
        tokens=tokens,
        morph_features=[],
        errors_span=[],
    )

test_sentences = [
    "ذهب الطالب الى المدرسة",
    "الطلاب يدرسون بجد",
    "قرأت الكتاب المفيد",
]

if best_model_path.exists():
    print("Testing inference...\n")
    print("="*60)
    
    for sentence in test_sentences:
        print(f"\nInput: {sentence}")
        try:
            gec_input = create_gec_input(sentence)
            predicted_labels = pipeline.predict(gec_input)
            print(f"Predicted: {predicted_labels}")
            
            words = sentence.split()
            edits = [(w, l) for w, l in zip(words, predicted_labels) if l not in ['K', 'K*']]
            if edits:
                print("Edits:")
                for word, label in edits:
                    print(f"  - {word} → {label}")
            else:
                print("No edits suggested")
        except Exception as e:
            print(f"Error: {e}")
        print("-"*60)

## Summary

Pipeline completed:
1. ✓ Loaded preprocessed training data
2. ✓ Trained BERT token classifier
3. ✓ Ran inference on test sentences

### Next Steps:
- Create proper dev/test splits
- Implement M² scorer evaluation
- Add edit rewriter to apply corrections
- Integrate with ontology/dictionary modules